In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import firedrake
from firedrake import inner, grad, dx, ds, dS, avg, jump, Constant, as_vector
import irksome
from irksome import Dt

In [ ]:
nz = 32
mesh = firedrake.UnitIntervalMesh(nz)

In [ ]:
element = firedrake.FiniteElement("CG", "interval", 1)
Q = firedrake.FunctionSpace(mesh, element)

In [ ]:
T_m = Constant(0.0)

T_1 = Constant(-1.0)
T_2 = Constant(+1.0)

In [ ]:
# TODO: Use real values
ρ_s = Constant(1.0)
ρ_l = Constant(1.0)
c_s = Constant(1.0)
c_l = Constant(1.0)

ρ = Constant(1.0)
c = Constant(1.0)

k_s = Constant(2.0)
k_l = Constant(1.0)

# TODO: rewrite this in terms of `k` and a skin depth `λ`
σ = Constant(1.0)

In [ ]:
T = firedrake.Function(Q)
T.assign(T_1)
ϕ = firedrake.TestFunction(Q)

k = firedrake.conditional(T < T_m, k_s, k_l)
F_cells = (ρ * c * Dt(T) * ϕ + k * inner(grad(T), grad(ϕ))) * dx
F_boundaries = σ * (T - T_1) * ϕ * ds((1,)) + σ * (T - T_2) * ϕ * ds((2,))

F = F_cells + F_boundaries

In [ ]:
final_time = 10.0
timestep = 1 / 16
num_steps = int(final_time / timestep)
dt = Constant(timestep)
t = Constant(0.0)

In [ ]:
params = {
    "solver_parameters": {
        "snes_type": "ksponly",
        "ksp_type": "gmres",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
        "snes_monitor": None,
    },
}

In [ ]:
method = irksome.BackwardEuler()
solver = irksome.TimeStepper(F, method, t, dt, T, **params)

In [ ]:
Ts = [T.copy(deepcopy=True)]

for step in range(num_steps):
    solver.advance()
    Ts.append(T.copy(deepcopy=True))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
firedrake.plot(T, axes=ax);